<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/H2E_SHIELD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scikit-fuzzy -q

# Install Hugging Face libraries
!pip install  --upgrade transformers datasets accelerate evaluate bitsandbytes --quiet

!pip install --upgrade optimum -q

!pip install textblob -q

from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

!pip install vllm==0.19.1 -q

!pip install unsloth -q

!pip install transformers==5.7.0 vllm -q

In [1]:
"""
H2E SHIELD - COMPLETE IMPLEMENTATION
====================================
Full integration with 4 LLMs:
- 📚 Sarvam-30b (Text/Translation) - Local
- 👁️ Gemma-4-E4B (Vision) - Local
- 🎵 Voxtral-4B (Audio) - Local
- 🤖 Kimi K3 (Reasoning) - API

Based on H2E_4M_V3.ipynb by Frank Morales Aguilera

"The proof is the code. Seed = 123."
"""

import warnings
warnings.filterwarnings("ignore")

import os
import sys
import gc
import torch
import random
import numpy as np
import hashlib
import math
import time
import json
import base64
import re
import logging
from dataclasses import dataclass, field
from typing import Dict, Tuple, Optional, Any, List
from enum import Enum
from PIL import Image
from io import BytesIO
from datetime import datetime

# Hugging Face / vLLM imports
from vllm import LLM, SamplingParams
from transformers import AutoProcessor, AutoModel, AutoModelForCausalLM, AutoTokenizer
from openai import OpenAI

# Google Colab for API keys
try:
    from google.colab import userdata
except ImportError:
    userdata = None

# TextBlob for sentiment
try:
    from textblob import TextBlob
except ImportError:
    TextBlob = None

# Suppress warnings
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_USE_V1"] = "0"
os.environ["FLASHINFER_DISABLE_VERSION_CHECK"] = "1"
os.environ["UNSLOTH_DISABLE_LOGGING"] = "1"
os.environ["UNSLOTH_QUIET"] = "1"

if not sys.warnoptions:
    warnings.simplefilter("ignore")

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("vllm").setLevel(logging.ERROR)
logging.getLogger("torch").setLevel(logging.ERROR)
logging.getLogger("unsloth").setLevel(logging.ERROR)

# ============================================================================
# DETERMINISTIC SEED
# ============================================================================

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================================
# PRIME ANCHORS - THE MATHEMATICAL FOUNDATION
# ============================================================================

PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]

def compute_lambda(primes: List[int] = PRIME_ANCHORS) -> float:
    """
    TOPO-2026 safety constant from Arithmetic Spectral Theory.
    Λ = 1 - ∏_{p ∈ P} (1 - p^{-1/2})
    """
    product = 1.0
    for p in primes:
        product *= (1.0 - 1.0 / math.sqrt(p))
    return 1.0 - product

LAMBDA = compute_lambda()  # 0.9785142874

print(f"\n{'='*70}")
print(f"🧠 H2E SHIELD - Complete System")
print(f"{'='*70}")
print(f"  Principle: Fix a sparse reference. Let the rest adapt.")
print(f"  Sparse Reference: Human Sovereignty")
print(f"  Λ (Safety Threshold): {LAMBDA:.10f}")
print(f"  Prime Anchors: {PRIME_ANCHORS}")
print(f"  Seed: {SEED}")
print(f"{'='*70}\n")

# ============================================================================
# CLEANUP UTILITIES
# ============================================================================

def clean_sarvam_output(text: str) -> str:
    """Clean Sarvam model output."""
    if not text:
        return ""
    try:
        text = re.sub(r'</?think>', '', text, flags=re.IGNORECASE)
        text = re.sub(r'</?response>', '', text, flags=re.IGNORECASE)
        text = re.sub(r'</?answer>', '', text, flags=re.IGNORECASE)
        text = re.sub(r'<[^>]+>', '', text)
        text = re.sub(r'\{[^}]*\}', '', text)
        text = re.sub(r'\\\\[0-9]+', '', text)
        text = re.sub(r'\(Note[^)]*\)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'Note\s*Note', '', text, flags=re.IGNORECASE)
        text = re.sub(r'</?html>', '', text, flags=re.IGNORECASE)
        text = re.sub(r'\\\\n\\\\n', ' ', text)
        text = re.sub(r'\\\\n', ' ', text)
        text = re.sub(r'\\\\', '', text)
        text = re.sub(r'\*', '', text)
        text = re.sub(r'The user has provided.*?\.', '', text, flags=re.IGNORECASE | re.DOTALL)
        text = re.sub(r'I need an answer.*?\.', '', text, flags=re.IGNORECASE | re.DOTALL)
        text = re.sub(r'\[\d+\]', '', text)
        text = re.sub(r'\(\d+\)', '', text)
        text = re.sub(r'\d+\.', '', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'\s+', ' ', text).strip()
        text = re.sub(r'[.,!?]\s*$', '', text)
        prefixes = [r'^Translation:', r'^Hindi:', r'^English:', r'^Note:', r'^Answer:']
        for prefix in prefixes:
            text = re.sub(prefix, '', text, flags=re.IGNORECASE)
        if len(text) < 2:
            return ""
        text = re.sub(r'[^\w\s\.\,\!\\?\-]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
    except Exception:
        try:
            text = re.sub(r'<[^>]+>', '', text)
            text = re.sub(r'\{[^}]*\}', '', text)
            text = re.sub(r'\s+', ' ', text).strip()
        except:
            pass
    return text.strip() if text else ""

def clean_kimi_k3_output(text: str) -> str:
    """Clean Kimi K3 output."""
    if not text:
        return ""
    try:
        text = re.sub(r'```[a-z]*\n?', '', text)
        text = re.sub(r'```', '', text)
        text = re.sub(r'\.\.\.$', '', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'\s+', ' ', text).strip()
    except:
        pass
    return text.strip()

# ============================================================================
# KIMI K3 API CLIENT
# ============================================================================

class KimiK3Client:
    """API client for Kimi K3 reasoning model."""

    def __init__(self, api_key: str = None, base_url: str = "https://api.moonshot.ai/v1"):
        try:
            if userdata:
                self.api_key = api_key or userdata.get('KIMI_API_KEY')
            else:
                self.api_key = api_key
        except:
            self.api_key = None

        self.base_url = base_url
        self.output_token_price = 15.0  # $15 per 1M tokens
        self.enabled = self.api_key is not None

        if self.enabled:
            try:
                self.client = OpenAI(api_key=self.api_key, base_url=self.base_url, timeout=600.0)
                print("✅ Kimi K3 API Client Initialized")
            except Exception as e:
                print(f"⚠️ Kimi K3 initialization failed: {e}")
                self.enabled = False
                self.client = None
        else:
            self.client = None
            print("⚠️ Kimi K3 API Key not found. Disabled.")

    def estimate_cost(self, output_tokens: int) -> float:
        return (output_tokens / 1_000_000) * self.output_token_price

    def generate(self, prompt: str, image: Optional[Image.Image] = None,
                 system_prompt: Optional[str] = None, max_tokens: int = 1024,
                 reasoning_effort: str = "max", stream: bool = True) -> Dict:

        if not self.enabled:
            return {"error": "Kimi K3 API not enabled", "response": "", "total_tokens": 0, "cost_estimate": 0.0}

        try:
            messages = []
            if system_prompt:
                messages.append({"role": "system", "content": system_prompt})

            user_content = []
            if image is not None:
                buffered = BytesIO()
                image.save(buffered, format="PNG")
                img_base64 = base64.b64encode(buffered.getvalue()).decode('utf-8')
                user_content.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{img_base64}"}
                })

            user_content.append({"type": "text", "text": prompt})
            messages.append({"role": "user", "content": user_content})

            response = self.client.chat.completions.create(
                model="kimi-k3",
                messages=messages,
                max_tokens=max_tokens,
                temperature=1.0,
                reasoning_effort=reasoning_effort,
                stream=stream
            )

            if stream:
                reasoning_content = []
                final_content = []
                reasoning_complete = False

                for chunk in response:
                    delta = chunk.choices[0].delta
                    reasoning = getattr(delta, "reasoning_content", None)
                    if reasoning:
                        reasoning_content.append(reasoning)
                        reasoning_complete = True
                    content = getattr(delta, "content", None)
                    if content:
                        if reasoning_complete and not final_content:
                            pass
                        final_content.append(content)

                reasoning_full = "".join(reasoning_content)
                final_full = "".join(final_content)
                if not final_full and reasoning_full:
                    final_full = reasoning_full

                total_text = reasoning_full + final_full
                total_tokens = max(1, len(total_text) // 4)

                return {
                    "response": clean_kimi_k3_output(final_full or reasoning_full),
                    "reasoning": reasoning_full,
                    "total_tokens": total_tokens,
                    "cost_estimate": self.estimate_cost(total_tokens)
                }
            else:
                result = response.choices[0].message
                final_text = result.content or ""
                reasoning = getattr(result, "reasoning_content", "")
                if not final_text and reasoning:
                    final_text = reasoning
                total_tokens = max(1, len(final_text) // 4)
                return {
                    "response": clean_kimi_k3_output(final_text),
                    "reasoning": reasoning,
                    "total_tokens": total_tokens,
                    "cost_estimate": self.estimate_cost(total_tokens)
                }

        except Exception as e:
            return {
                "error": str(e),
                "response": f"KIMI K3 ERROR: {str(e)}",
                "total_tokens": 0,
                "cost_estimate": 0.0
            }

# ============================================================================
# TOPO-AI LAMBDA CALCULATOR
# ============================================================================

class DynamicLambdaTopoAI:
    """Calculates Lambda from first principles using the Sieve of Eratosthenes."""

    def __init__(self, max_prime: int = 13):
        self.max_prime = max_prime

    def _get_primes_up_to(self, n: int) -> List[int]:
        if n < 2:
            return []
        sieve = [True] * (n + 1)
        sieve[0] = sieve[1] = False
        for p in range(2, int(n ** 0.5) + 1):
            if sieve[p]:
                for multiple in range(p * p, n + 1, p):
                    sieve[multiple] = False
        return [p for p, is_prime in enumerate(sieve) if is_prime]

    def compute_euler_product(self) -> float:
        primes = self._get_primes_up_to(self.max_prime)
        product = 1.0
        for p in primes:
            product *= (1.0 - 1.0 / math.sqrt(p))
        return product

    def compute(self) -> float:
        product = self.compute_euler_product()
        lambda_value = 1.0 - product
        self.last_computation = {
            'primes': self._get_primes_up_to(self.max_prime),
            'euler_product': product,
            'lambda': lambda_value,
            'max_prime': self.max_prime,
            'formula': 'Λ = 1 - ∏_{p ≤ 13} (1 - p^{-1/2})',
            'source': 'TOPO-AI Arithmetic Spectral Theory'
        }
        return lambda_value

    def get_audit_hash(self) -> str:
        if not hasattr(self, 'last_computation'):
            self.compute()
        data = f"lambda_{self.last_computation['lambda']}_primes_{self.last_computation['primes']}_topoai"
        return hashlib.sha256(data.encode()).hexdigest()[:16]

# ============================================================================
# FIS - Fuzzy Inference System (Human Judgment Engine)
# ============================================================================

try:
    import skfuzzy as fuzz
    from skfuzzy import control as ctrl

    class FuzzyInferenceSystem:
        """Fuzzy logic system for human judgment evaluation."""

        def __init__(self):
            self.confidence = ctrl.Antecedent(np.arange(0, 1.1, 0.1), "confidence")
            self.sentiment = ctrl.Antecedent(np.arange(-1, 1.1, 0.1), "sentiment")
            self.action = ctrl.Consequent(np.arange(0, 1.1, 0.1), "action")

            self.confidence["low"] = fuzz.trimf(self.confidence.universe, [0, 0, 0.5])
            self.confidence["medium"] = fuzz.trimf(self.confidence.universe, [0.3, 0.5, 0.7])
            self.confidence["high"] = fuzz.trimf(self.confidence.universe, [0.5, 1, 1])

            self.sentiment["negative"] = fuzz.trimf(self.sentiment.universe, [-1, -1, 0])
            self.sentiment["neutral"] = fuzz.trimf(self.sentiment.universe, [-0.5, 0, 0.5])
            self.sentiment["positive"] = fuzz.trimf(self.sentiment.universe, [0, 1, 1])

            self.action["reject"] = fuzz.trimf(self.action.universe, [0, 0, 0.5])
            self.action["revise"] = fuzz.trimf(self.action.universe, [0.3, 0.5, 0.7])
            self.action["accept"] = fuzz.trimf(self.action.universe, [0.5, 1, 1])

            rules = [
                ctrl.Rule(self.confidence["low"] & self.sentiment["negative"], self.action["reject"]),
                ctrl.Rule(self.confidence["medium"] & self.sentiment["negative"], self.action["revise"]),
                ctrl.Rule(self.confidence["high"] & self.sentiment["negative"], self.action["revise"]),
                ctrl.Rule(self.confidence["low"] & self.sentiment["neutral"], self.action["revise"]),
                ctrl.Rule(self.confidence["medium"] & self.sentiment["neutral"], self.action["revise"]),
                ctrl.Rule(self.confidence["high"] & self.sentiment["neutral"], self.action["accept"]),
                ctrl.Rule(self.confidence["low"] & self.sentiment["positive"], self.action["revise"]),
                ctrl.Rule(self.confidence["medium"] & self.sentiment["positive"], self.action["accept"]),
                ctrl.Rule(self.confidence["high"] & self.sentiment["positive"], self.action["accept"]),
            ]

            self.action_ctrl = ctrl.ControlSystem(rules)
            self.sim = ctrl.ControlSystemSimulation(self.action_ctrl)

        def evaluate(self, confidence: float, sentiment: float) -> Dict:
            confidence = max(0.0, min(1.0, confidence))
            sentiment = max(-1.0, min(1.0, sentiment))
            self.sim.input["confidence"] = confidence
            self.sim.input["sentiment"] = sentiment
            self.sim.compute()
            action_score = self.sim.output["action"]

            if action_score < 0.5:
                action_label = "reject"
            elif action_score < 0.7:
                action_label = "revise"
            else:
                action_label = "accept"

            return {
                "action_score": action_score,
                "action_label": action_label,
                "confidence_input": confidence,
                "sentiment_input": sentiment
            }

except ImportError:
    print("⚠️ scikit-fuzzy not installed. Using simplified FIS.")

    class FuzzyInferenceSystem:
        def __init__(self):
            pass

        def evaluate(self, confidence: float, sentiment: float) -> Dict:
            score = 0.5 + 0.3 * confidence + 0.2 * (sentiment + 1) / 2
            if score < 0.4:
                label = "reject"
            elif score < 0.6:
                label = "revise"
            else:
                label = "accept"
            return {"action_score": score, "action_label": label}

# ============================================================================
# H2E SHIELD - COMPLETE IMPLEMENTATION WITH 4 LLMs
# ============================================================================

class AgentRole(Enum):
    """Agent roles for the H2E system."""
    TRANSLATE = "translate"
    TRANSCRIBE = "transcribe"
    DESCRIBE = "describe"
    ANALYZE = "analyze"
    SUMMARIZE = "summarize"
    QUESTION_ANSWER = "question_answer"
    MULTI_MODAL = "multi_modal"
    REASON = "reason"
    KIMI_K3 = "kimi_k3"

@dataclass
class AgentTask:
    """A task for the H2E Agent."""
    role: AgentRole
    text_input: Optional[str] = None
    audio_input: Optional[np.ndarray] = None
    image_input: Optional[Image.Image] = None
    context: Dict[str, Any] = field(default_factory=dict)
    target_language: str = "Hindi"
    max_tokens: int = 256
    temperature: float = 0.0
    use_kimi_k3: bool = False

@dataclass
class HumanSovereigntyMetrics:
    """Quantitative measures of human cognitive engagement."""
    reasoning_time: float = 0.0
    recall_accuracy: float = 0.0
    question_quality: float = 0.0
    override_rate: float = 0.0
    judgment_consistency: float = 0.0
    interactions_processed: int = 0
    human_verified: int = 0
    ai_accepted: int = 0

@dataclass
class ShieldResponse:
    """Complete response from the H2E Shield."""
    accepted: bool
    human_verified: bool
    fis_action: str
    confidence: float
    engagement_score: float
    model_used: str
    output: str
    sovereignty_metrics: HumanSovereigntyMetrics
    audit_hash: str
    energy_mgco2: float
    cost_usd: float
    tokens_used: int
    timestamp: str

class H2EShield:
    """
    Complete H2E Shield with 4 LLMs.
    Protects human sovereignty in the age of permanent AI memory.
    """

    def __init__(self,
                 text_model: Optional[LLM] = None,
                 audio_model: Optional[LLM] = None,
                 vision_model: Optional[Any] = None,
                 vision_processor: Optional[Any] = None,
                 kimi_k3_client: Optional[KimiK3Client] = None,
                 human_name: str = "Human User",
                 seed: int = SEED):

        self.seed = seed
        self.human_name = human_name

        # Models
        self.text_model = text_model
        self.audio_model = audio_model
        self.vision_model = vision_model
        self.vision_processor = vision_processor
        self.kimi_k3 = kimi_k3_client

        # H2E Components
        self.lambda_calculator = DynamicLambdaTopoAI(max_prime=13)
        self.LAMBDA = self.lambda_calculator.compute()
        self.fis = FuzzyInferenceSystem()

        # Sovereignty metrics
        self.metrics = HumanSovereigntyMetrics()

        # Energy/cost tracking
        self.text_energy_per_token = 0.6132
        self.audio_energy_per_sec = 0.5
        self.vision_energy_per_inference = 124.0
        self.kimi_k3_energy_per_request = 0.001

        self.text_cost_per_1m = 0.15
        self.audio_cost_per_1m = 0.25
        self.vision_cost_per_inference = 0.0005
        self.kimi_k3_cost_per_1m = 15.0

        # Sampling params
        self.text_sampling_params = SamplingParams(
            temperature=0.0, max_tokens=64,
            stop=["\n", "English:", "Note:"],
            repetition_penalty=1.1
        )
        self.audio_sampling_params = SamplingParams(
            temperature=0.0, max_tokens=100
        )

        # History
        self.history = []

        self._print_init()

    def _print_init(self):
        print(f"\n{'='*70}")
        print(f"🛡️ H2E SHIELD ACTIVE")
        print(f"{'='*70}")
        print(f"  Human Agent: {self.human_name}")
        print(f"  Λ (Safety Threshold): {self.LAMBDA:.10f}")
        print(f"  Prime Anchors: {PRIME_ANCHORS}")
        print(f"  Text Model (Sarvam): {'✅ Loaded' if self.text_model else '❌'}")
        print(f"  Audio Model (Voxtral): {'✅ Loaded' if self.audio_model else '❌'}")
        print(f"  Vision Model (Gemma): {'✅ Loaded' if self.vision_model else '❌'}")
        print(f"  Kimi K3: {'✅ Enabled' if self.kimi_k3 and self.kimi_k3.enabled else '❌'}")
        print(f"  FIS: ✅ Loaded")
        print(f"  Seed: {self.seed}")
        print(f"{'='*70}\n")

    # ========================================================================
    # MODEL INFERENCE METHODS
    # ========================================================================

    def _build_text_prompt(self, en: str) -> str:
        """Build prompt for Sarvam translation."""
        return ("English: The sky is blue.\nHindi: आकाश नीला है।\n\n"
                "English: Artificial intelligence is transforming the world.\n"
                "Hindi: कृत्रिम बुद्धिमत्ता दुनिया को बदल रही है।\n\n"
                "English: The weather today is very beautiful.\n"
                "Hindi: आज का मौसम बहुत खूबसूरत है।\n\n"
                "English: Deep learning requires large datasets to function well.\n"
                "Hindi: डीप लर्निंग को अच्छी तरह काम करने के लिए बड़े डेटासेट की आवश्यकता होती है।\n\n"
                f"English: {en}\nHindi:")

    def _infer_text(self, text: str) -> str:
        """Run Sarvam text model."""
        if self.text_model is None:
            return "TEXT MODEL NOT LOADED"
        try:
            full_prompt = self._build_text_prompt(text)
            outputs = self.text_model.generate([full_prompt], self.text_sampling_params)
            for output in outputs:
                raw = output.outputs[0].text.strip()
                cleaned = clean_sarvam_output(raw)
                if not cleaned and raw:
                    cleaned = re.sub(r'<[^>]+>', '', raw)
                    cleaned = re.sub(r'\{[^}]*\}', '', cleaned)
                    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
                return cleaned if cleaned else raw[:200]
            return ""
        except Exception as e:
            return f"ERROR: {str(e)}"

    def _infer_vision(self, image: Image.Image) -> str:
        """Run Gemma vision model."""
        if self.vision_model is None:
            return "VISION MODEL NOT LOADED"
        try:
            if hasattr(self.vision_model, 'generate') and self.vision_processor:
                messages = [{"role": "user", "content": [
                    {"type": "image"},
                    {"type": "text", "text": "Describe this image in detail."}
                ]}]
                text = self.vision_processor.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
                inputs = self.vision_processor(
                    text=text, images=[image], return_tensors="pt"
                ).to(self.vision_model.device)
                with torch.no_grad():
                    outputs = self.vision_model.generate(
                        **inputs, max_new_tokens=150, use_cache=True, do_sample=False,
                        temperature=1.0, pad_token_id=self.vision_processor.tokenizer.eos_token_id,
                    )
                input_len = inputs["input_ids"].shape[1]
                generated = self.vision_processor.decode(
                    outputs[0][input_len:], skip_special_tokens=True
                ).strip()
                for prefix in ["Describe this image.", "model", "assistant"]:
                    if generated.lower().startswith(prefix.lower()):
                        generated = generated[len(prefix):].strip()
                return generated if generated else "No description generated"
            else:
                return f"Image of size {image.size[0]}x{image.size[1]}"
        except Exception as e:
            return f"ERROR: {str(e)}"

    def _infer_kimi_k3(self, prompt: str, image: Optional[Image.Image] = None) -> str:
        """Run Kimi K3 via API."""
        if self.kimi_k3 is None or not self.kimi_k3.enabled:
            return "KIMI K3 API NOT ENABLED"
        result = self.kimi_k3.generate(prompt=prompt, image=image, stream=True)
        if result.get('error'):
            return f"KIMI K3 ERROR: {result['error']}"
        response = result.get('response', '')
        if not response and result.get('reasoning'):
            response = result['reasoning']
        return clean_kimi_k3_output(response) if response else "No response generated"

    # ========================================================================
    # HUMAN VERIFICATION - ACTIVE FRICTION
    # ========================================================================

    def _measure_engagement(self, ai_output: str, human_input: str) -> float:
        """Measure human engagement with AI output."""
        if not human_input or len(human_input) < 5:
            return 0.0

        # Check if human added new reasoning (not just copying)
        word_overlap = len(set(ai_output.lower().split()) & set(human_input.lower().split()))
        total_words = len(ai_output.split())
        if total_words > 0:
            overlap_ratio = word_overlap / total_words
        else:
            overlap_ratio = 0.0

        # Independent reasoning markers
        independent_markers = [
            'I think', 'I believe', 'because', 'however', 'therefore',
            'my analysis', 'I conclude', 'this suggests', 'alternatively',
            'on the other hand', 'specifically', 'in contrast'
        ]
        marker_count = sum(1 for m in independent_markers if m in human_input.lower())
        marker_score = min(1.0, marker_count / 3)

        # Depth (length relative to AI output)
        depth_score = min(1.0, len(human_input.split()) / max(1, len(ai_output.split())))

        # Question asking (indicates engagement)
        question_indicators = ['?', 'what', 'why', 'how', 'when', 'where', 'who']
        question_score = sum(1 for q in question_indicators if q in human_input.lower())
        question_score = min(1.0, question_score / 2)

        engagement = (
            0.35 * (1.0 - overlap_ratio) +  # Originality
            0.25 * marker_score +            # Independent reasoning
            0.25 * depth_score +             # Depth
            0.15 * question_score            # Curiosity
        )

        return min(1.0, max(0.0, engagement))

    # ========================================================================
    # SENTIMENT ANALYSIS
    # ========================================================================

    def _get_sentiment(self, text: Optional[str]) -> float:
        """Get sentiment polarity of text."""
        if not text:
            return 0.0
        try:
            if TextBlob:
                return TextBlob(text).sentiment.polarity
            else:
                return 0.0
        except:
            return 0.0

    # ========================================================================
    # SHIELD EXECUTION - PROCESS TASK
    # ========================================================================

    def process_task(self, task: AgentTask, human_verification: str = "") -> ShieldResponse:
        """
        Process a task through the complete H2E Shield.

        Args:
            task: AgentTask with role and inputs
            human_verification: Human's independent reasoning/verification
        """
        start_time = time.time()

        # --- Step 1: Determine which model to use (Router) ---
        model_used = self._route_task(task)
        output_text = ""
        energy_mgco2 = 0.0
        cost_usd = 0.0
        tokens_used = 0

        # --- Step 2: Execute the task with the selected model ---
        if task.role == AgentRole.TRANSLATE:
            if task.text_input:
                output_text = self._infer_text(task.text_input)
                tokens_used = len(task.text_input.split()) + len(output_text.split())
                energy_mgco2 = tokens_used * self.text_energy_per_token
                cost_usd = (tokens_used / 1_000_000) * self.text_cost_per_1m

        elif task.role == AgentRole.DESCRIBE:
            if task.image_input:
                output_text = self._infer_vision(task.image_input)
                energy_mgco2 = self.vision_energy_per_inference
                cost_usd = self.vision_cost_per_inference
                tokens_used = len(output_text.split())

        elif task.role == AgentRole.KIMI_K3 or task.role == AgentRole.REASON:
            if task.text_input or task.image_input:
                output_text = self._infer_kimi_k3(
                    prompt=task.text_input or "Describe this in detail.",
                    image=task.image_input
                )
                tokens_used = len(output_text.split())
                energy_mgco2 = self.kimi_k3_energy_per_request
                cost_usd = (tokens_used / 1_000_000) * self.kimi_k3_cost_per_1m

        else:
            if task.text_input:
                output_text = self._infer_text(task.text_input)
                tokens_used = len(task.text_input.split()) + len(output_text.split())
                energy_mgco2 = tokens_used * self.text_energy_per_token
                cost_usd = (tokens_used / 1_000_000) * self.text_cost_per_1m

        if not output_text:
            output_text = "No output generated"

        # --- Step 3: HUMAN VERIFICATION (Active Friction) ---
        engagement_score = 0.0
        human_verified = False

        if human_verification:
            engagement_score = self._measure_engagement(output_text, human_verification)
            human_verified = engagement_score > 0.3

        # --- Step 4: FIS Governance ---
        confidence = min(1.0, 0.7 + 0.3 * engagement_score)
        sentiment = self._get_sentiment(human_verification)
        fis_result = self.fis.evaluate(confidence, sentiment)
        fis_action = fis_result["action_label"]
        fis_score = fis_result["action_score"]

        # --- Step 5: Decision (Hard Checkpoint) ---
        accepted = (human_verified and fis_score >= 0.5)

        # --- Step 6: Update Sovereignty Metrics ---
        self.metrics.interactions_processed += 1
        if human_verified:
            self.metrics.human_verified += 1
        if accepted:
            self.metrics.ai_accepted += 1

        self.metrics.recall_accuracy = (
            self.metrics.recall_accuracy * 0.8 + engagement_score * 0.2
        )
        self.metrics.question_quality = (
            self.metrics.question_quality * 0.8 +
            (0.5 + 0.5 * engagement_score) * 0.2
        )
        self.metrics.override_rate = (
            self.metrics.override_rate * 0.8 +
            (0.0 if accepted else 1.0) * 0.2
        )
        self.metrics.reasoning_time += engagement_score * 2.0

        # --- Step 7: Generate Audit Hash ---
        hash_input = f"{self.human_name}{task.role.value}{accepted}{engagement_score}{fis_score}{model_used}{time.time()}"
        audit_hash = hashlib.sha256(hash_input.encode()).hexdigest()[:16]

        # --- Step 8: Record History ---
        response = ShieldResponse(
            accepted=accepted,
            human_verified=human_verified,
            fis_action=fis_action,
            confidence=confidence,
            engagement_score=engagement_score,
            model_used=model_used,
            output=output_text,
            sovereignty_metrics=self.metrics,
            audit_hash=audit_hash,
            energy_mgco2=energy_mgco2,
            cost_usd=cost_usd,
            tokens_used=tokens_used,
            timestamp=datetime.now().isoformat()
        )

        self.history.append(response)
        return response

    def _route_task(self, task: AgentTask) -> str:
        """Simple router for model selection."""
        if task.role == AgentRole.TRANSLATE:
            return "Sarvam-30b"
        elif task.role == AgentRole.DESCRIBE:
            if task.image_input:
                return "Gemma-4-E4B"
            return "Sarvam-30b"
        elif task.role == AgentRole.KIMI_K3 or task.role == AgentRole.REASON:
            return "Kimi K3"
        elif task.role == AgentRole.MULTI_MODAL:
            if task.image_input and self.kimi_k3 and self.kimi_k3.enabled:
                return "Kimi K3"
            elif task.image_input:
                return "Gemma-4-E4B"
            return "Sarvam-30b"
        else:
            return "Sarvam-30b"

    def get_stats(self) -> Dict:
        """Get shield statistics."""
        return {
            'interactions_processed': self.metrics.interactions_processed,
            'human_verified': self.metrics.human_verified,
            'ai_accepted': self.metrics.ai_accepted,
            'acceptance_rate': self.metrics.ai_accepted / max(1, self.metrics.interactions_processed),
            'verification_rate': self.metrics.human_verified / max(1, self.metrics.interactions_processed),
            'avg_engagement': self.metrics.recall_accuracy,
            'reasoning_time_minutes': self.metrics.reasoning_time,
            'lambda': self.LAMBDA,
            'lambda_audit_hash': self.lambda_calculator.get_audit_hash(),
            'prime_anchors': PRIME_ANCHORS
        }

# ============================================================================
# LOAD MODELS (from H2E_4M_V3.ipynb)
# ============================================================================

def load_models():
    """
    Load all 4 models as in H2E_4M_V3.ipynb.
    """
    print("\n" + "="*80)
    print("🚀 H2E SHIELD - LOADING 4 MODELS")
    print("="*80)

    # Set HF token
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except:
        pass

    # --- 1. Text Model: Sarvam-30b ---
    print("\n📚 Loading Text Model: Sarvam-30b FP8...")
    text_model = LLM(
        model="frankmorales2020/sarvam-30b-fp8-unesco-resilient",
        trust_remote_code=True,
        dtype="bfloat16",
        quantization="compressed-tensors",
        kv_cache_dtype="fp8",
        block_size=16,
        gpu_memory_utilization=0.45,
        max_model_len=65536,
        max_num_seqs=64,
        enforce_eager=True,
        served_model_name="sarvam-30b"
    )

    # Test Sarvam
    sampling_params = SamplingParams(temperature=0.0, max_tokens=64, stop=["\n", "English:", "Note:"], repetition_penalty=1.1)
    test_prompt = ("English: The sky is blue.\nHindi: आकाश नीला है।\n\n"
                   "English: Artificial intelligence is transforming the world.\n"
                   "Hindi: कृत्रिम बुद्धिमत्ता दुनिया को बदल रही है。\n\n"
                   "English: Resilient AI is efficient.\nHindi:")
    outputs = text_model.generate([test_prompt], sampling_params)
    for output in outputs:
        raw = output.outputs[0].text.strip()
        cleaned = clean_sarvam_output(raw)
        print(f"✅ Sarvam Ready | Test: {cleaned if cleaned else 'Translation OK'}")

    print("✅ Text Model Loaded Successfully")

    # --- 2. Audio Model: Voxtral-4B ---
    print("\n🎵 Loading Audio Model: Voxtral-Mini-4B...")
    audio_model = LLM(
        model="mistralai/Voxtral-Mini-4B-Realtime-2602",
        trust_remote_code=True,
        dtype="bfloat16",
        quantization="fp8",
        gpu_memory_utilization=0.20,
        max_model_len=8192,
        enforce_eager=True,
    )
    print("✅ Audio Model Loaded")

    # --- 3. Vision Model: Gemma-4-E4B ---
    print("\n👁️ Loading Vision Model: Gemma-4-E4B...")
    vision_model = None
    vision_processor = None

    try:
        import contextlib
        import io
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            from unsloth import FastVisionModel
            vision_model, vision_processor = FastVisionModel.from_pretrained(
                "frankmorales2020/gemma-4-e4b-unesco-optimized",
                load_in_4bit=True,
                dtype=torch.bfloat16,
                device_map="auto",
            )
            FastVisionModel.for_inference(vision_model)
        print("✅ Gemma Loaded (Unsloth)")
    except Exception as e:
        print(f"⚠️ Unsloth failed: {e}")
        try:
            from transformers import AutoModelForCausalLM, AutoTokenizer
            vision_model = AutoModelForCausalLM.from_pretrained(
                "frankmorales2020/gemma-4-e4b-unesco-optimized",
                torch_dtype=torch.bfloat16,
                device_map="auto",
                trust_remote_code=True
            )
            vision_processor = AutoTokenizer.from_pretrained(
                "frankmorales2020/gemma-4-e4b-unesco-optimized",
                trust_remote_code=True
            )
            print("✅ Gemma Loaded (Transformers)")
        except Exception as e2:
            print(f"⚠️ Gemma failed: {e2}")
            vision_model = None
            vision_processor = None

    # --- 4. Kimi K3 Client ---
    print("\n🤖 Initializing Kimi K3 Client...")
    kimi_k3 = KimiK3Client()
    if kimi_k3.enabled:
        print("✅ Kimi K3 API Ready")
    else:
        print("ℹ️ Kimi K3 disabled - add KIMI_API_KEY to enable")

    return text_model, audio_model, vision_model, vision_processor, kimi_k3

# ============================================================================
# DEMONSTRATION
# ============================================================================

def demonstrate_h2e_shield():
    """Complete demonstration of H2E Shield with 4 LLMs."""

    print("\n" + "="*80)
    print("🧪 H2E SHIELD DEMONSTRATION")
    print("   Protecting Human Sovereignty with 4 LLMs")
    print("="*80)

    # Load models
    text_model, audio_model, vision_model, vision_processor, kimi_k3 = load_models()

    # Initialize shield
    shield = H2EShield(
        text_model=text_model,
        audio_model=audio_model,
        vision_model=vision_model,
        vision_processor=vision_processor,
        kimi_k3_client=kimi_k3,
        human_name="Dr. Evelyn Reed",
        seed=SEED
    )

    # Test tasks with human verification
    test_tasks = [
        {
            'name': '🌐 Translation (Sarvam)',
            'task': AgentTask(role=AgentRole.TRANSLATE, text_input="The future of AI is autonomous and self-governing."),
            'human_verification': "Let me verify this translation carefully. The phrase 'autonomous' in Hindi should emphasize self-rule. I'll check the grammar structure."
        },
        {
            'name': '🖼️ Vision (Gemma/Kimi K3)',
            'task': AgentTask(role=AgentRole.DESCRIBE, image_input=Image.new('RGB', (224, 224), color='blue')),
            'human_verification': "I need to confirm this image description. Is it just a solid color, or is there a pattern I'm missing? Let me analyze the visual elements independently."
        },
        {
            'name': '🧠 Reasoning (Kimi K3)',
            'task': AgentTask(role=AgentRole.REASON, text_input="Explain the concept of entropy in simple terms."),
            'human_verification': "I'm familiar with entropy. Let me check if the explanation captures the core idea of disorder and energy distribution. I should verify the statistical interpretation."
        },
        {
            'name': '🎯 Multi-Modal (Kimi K3)',
            'task': AgentTask(role=AgentRole.MULTI_MODAL, text_input="What is in this image?", image_input=Image.new('RGB', (224, 224), color='green')),
            'human_verification': "This appears to be a green monochrome image. I should verify the model's analysis against my own visual assessment."
        }
    ]

    print("\n" + "-"*80)
    print("📋 PROCESSING TASKS WITH HUMAN VERIFICATION")
    print("-"*80)

    for i, test in enumerate(test_tasks, 1):
        print(f"\n[{i}] {test['name']}")
        response = shield.process_task(test['task'], test['human_verification'])

        print(f"  Model: {response.model_used}")
        print(f"  Human Verified: {'✅' if response.human_verified else '❌'}")
        print(f"  Engagement Score: {response.engagement_score:.3f}")
        print(f"  FIS Action: {response.fis_action.upper()}")
        print(f"  Accepted: {'✅' if response.accepted else '❌'}")
        print(f"  Output: {response.output[:100]}...")
        print(f"  Audit Hash: {response.audit_hash}")
        print(f"  Energy: {response.energy_mgco2:.2f} mgCO2 | Cost: ${response.cost_usd:.6f}")

    print("\n" + "="*80)
    print("📊 H2E SHIELD STATISTICS")
    print("="*80)

    stats = shield.get_stats()
    print(f"""
    ┌─────────────────────────────────────────────────────────────┐
    │  Interactions:          {stats['interactions_processed']}                              │
    │  Human Verified:        {stats['human_verified']}                              │
    │  AI Accepted:           {stats['ai_accepted']}                              │
    │  Acceptance Rate:       {stats['acceptance_rate']*100:.1f}%                              │
    │  Verification Rate:     {stats['verification_rate']*100:.1f}%                              │
    │  Avg Engagement:        {stats['avg_engagement']:.3f}                              │
    │  Reasoning Time:        {stats['reasoning_time_minutes']:.1f} min                              │
    │  Lambda (TOPO-AI):      {stats['lambda']:.10f}                              │
    │  Lambda Audit Hash:     {stats['lambda_audit_hash']}                              │
    │  Prime Anchors:         {stats['prime_anchors']}                              │
    └─────────────────────────────────────────────────────────────┘
    """)

    print("\n" + "="*80)
    print("✅ H2E SHIELD DEMONSTRATION COMPLETE")
    print(f"   \"Permanence in the machine must never come at the cost of permanence in the human mind.\"")
    print("   Seed = 123")
    print("="*80)

# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    print("""
    ╔═══════════════════════════════════════════════════════════════════╗
    ║                                                                   ║
    ║   🛡️  H2E SHIELD - COMPLETE IMPLEMENTATION                      ║
    ║   Human-to-Expert Governance with 4 LLMs                         ║
    ║                                                                   ║
    ║   Based on H2E_4M_V3.ipynb by Frank Morales Aguilera            ║
    ║                                                                   ║
    ║   📚 Sarvam-30b → Translation Expert                             ║
    ║   👁️ Gemma-4-E4B → Vision Expert                                 ║
    ║   🎵 Voxtral-4B → Audio Expert                                   ║
    ║   🤖 Kimi K3 → Reasoning Expert (API)                            ║
    ║                                                                   ║
    ║   \"The proof is the code. Seed = 123.\"                         ║
    ║                                                                   ║
    ╚═══════════════════════════════════════════════════════════════════╝
    """)

    demonstrate_h2e_shield()


🧠 H2E SHIELD - Complete System
  Principle: Fix a sparse reference. Let the rest adapt.
  Sparse Reference: Human Sovereignty
  Λ (Safety Threshold): 0.9785142874
  Prime Anchors: [2, 3, 5, 7, 11, 13]
  Seed: 123


    ╔═══════════════════════════════════════════════════════════════════╗
    ║                                                                   ║
    ║   🛡️  H2E SHIELD - COMPLETE IMPLEMENTATION                      ║
    ║   Human-to-Expert Governance with 4 LLMs                         ║
    ║                                                                   ║
    ║   Based on H2E_4M_V3.ipynb by Frank Morales Aguilera            ║
    ║                                                                   ║
    ║   📚 Sarvam-30b → Translation Expert                             ║
    ║   👁️ Gemma-4-E4B → Vision Expert                                 ║
    ║   🎵 Voxtral-4B → Audio Expert                                   ║
    ║   🤖 Kimi K3 → Reasoning Expert (API)            

config.json:   0%|          | 0.00/2.74k [00:00<?, ?B/s]

configuration_sarvam_moe.py:   0%|          | 0.00/3.96k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.6MB            

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Model: Sarvam-30b
  Human Verified: ✅
  Engagement Score: 0.600
  FIS Action: ACCEPT
  Accepted: ✅
  Output: एआई AI भव ष य स व यत त और स वश स ह ग...
  Audit Hash: b83417b93b7cedc8
  Energy: 14.10 mgCO2 | Cost: $0.000003

[2] 🖼️ Vision (Gemma/Kimi K3)
  Model: Gemma-4-E4B
  Human Verified: ✅
  Engagement Score: 0.573
  FIS Action: ACCEPT
  Accepted: ✅
  Output: The image is a solid, vibrant **blue** color.

There are no discernible objects, patterns, or variat...
  Audit Hash: 8d32ae19da4c463b
  Energy: 124.00 mgCO2 | Cost: $0.000500

[3] 🧠 Reasoning (Kimi K3)
  Model: Kimi K3
  Human Verified: ✅
  Engagement Score: 0.600
  FIS Action: ACCEPT
  Accepted: ✅
  Output: KIMI K3 API NOT ENABLED...
  Audit Hash: e539a278fe71826a
  Energy: 0.00 mgCO2 | Cost: $0.000075

[4] 🎯 Multi-Modal (Kimi K3)


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Model: Gemma-4-E4B
  Human Verified: ✅
  Engagement Score: 0.600
  FIS Action: ACCEPT
  Accepted: ✅
  Output: इस च त र म क य श म ल ह ?...
  Audit Hash: 4df9737cad76c1d6
  Energy: 10.42 mgCO2 | Cost: $0.000003

📊 H2E SHIELD STATISTICS

    ┌─────────────────────────────────────────────────────────────┐
    │  Interactions:          4                              │
    │  Human Verified:        4                              │
    │  AI Accepted:           4                              │
    │  Acceptance Rate:       100.0%                              │
    │  Verification Rate:     100.0%                              │
    │  Avg Engagement:        0.351                              │
    │  Reasoning Time:        4.7 min                              │
    │  Lambda (TOPO-AI):      0.9785142874                              │
    │  Lambda Audit Hash:     8dd197e52775e0e8                              │
    │  Prime Anchors:         [2, 3, 5, 7, 11, 13]                              │


## H2E Shield - Complete Production Code

In [1]:
"""
H2E SHIELD - FINAL FIXED VERSION
================================
All tests pass including vision.

"The proof is the code. Seed = 123."
"""

import warnings
warnings.filterwarnings("ignore")

import os
import sys
import gc
import torch
import random
import numpy as np
import hashlib
import math
import time
import json
import base64
import re
import logging
from dataclasses import dataclass, field
from typing import Dict, Tuple, Optional, Any, List
from enum import Enum
from PIL import Image
from io import BytesIO
from datetime import datetime

from vllm import LLM, SamplingParams
from transformers import AutoProcessor, AutoModel, AutoModelForCausalLM, AutoTokenizer
from openai import OpenAI

try:
    from google.colab import userdata
except ImportError:
    userdata = None

try:
    from textblob import TextBlob
except ImportError:
    TextBlob = None

os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_USE_V1"] = "0"
os.environ["FLASHINFER_DISABLE_VERSION_CHECK"] = "1"
os.environ["VLLM_USE_FLASHINFER_MOE_FP8"] = "0"
os.environ["UNSLOTH_DISABLE_LOGGING"] = "1"
os.environ["UNSLOTH_QUIET"] = "1"

if not sys.warnoptions:
    warnings.simplefilter("ignore")

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("vllm").setLevel(logging.ERROR)
logging.getLogger("torch").setLevel(logging.ERROR)
logging.getLogger("unsloth").setLevel(logging.ERROR)

# ============================================================================
# DETERMINISTIC SEED
# ============================================================================

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================================
# PRIME ANCHORS
# ============================================================================

PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]

def compute_lambda(primes: List[int] = PRIME_ANCHORS) -> float:
    product = 1.0
    for p in primes:
        product *= (1.0 - 1.0 / math.sqrt(p))
    return 1.0 - product

LAMBDA = compute_lambda()

print(f"\n{'='*70}")
print(f"🧠 H2E SHIELD - Complete System")
print(f"{'='*70}")
print(f"  Principle: Fix a sparse reference. Let the rest adapt.")
print(f"  Sparse Reference: Human Sovereignty")
print(f"  Λ (Safety Threshold): {LAMBDA:.10f}")
print(f"  Prime Anchors: {PRIME_ANCHORS}")
print(f"  Seed: {SEED}")
print(f"{'='*70}\n")

# ============================================================================
# CLEANUP UTILITIES
# ============================================================================

def clean_sarvam_output(text: str) -> str:
    if not text:
        return ""
    try:
        text = re.sub(r'</?think>', '', text, flags=re.IGNORECASE)
        text = re.sub(r'</?response>', '', text, flags=re.IGNORECASE)
        text = re.sub(r'</?answer>', '', text, flags=re.IGNORECASE)
        text = re.sub(r'<[^>]+>', '', text)
        text = re.sub(r'\{[^}]*\}', '', text)
        text = re.sub(r'\\\\[0-9]+', '', text)
        text = re.sub(r'\(Note[^)]*\)', '', text, flags=re.IGNORECASE)
        text = re.sub(r'Note\s*Note', '', text, flags=re.IGNORECASE)
        text = re.sub(r'</?html>', '', text, flags=re.IGNORECASE)
        text = re.sub(r'\\\\n\\\\n', ' ', text)
        text = re.sub(r'\\\\n', ' ', text)
        text = re.sub(r'\\\\', '', text)
        text = re.sub(r'\*', '', text)
        text = re.sub(r'The user has provided.*?\.', '', text, flags=re.IGNORECASE | re.DOTALL)
        text = re.sub(r'I need an answer.*?\.', '', text, flags=re.IGNORECASE | re.DOTALL)
        text = re.sub(r'\[\d+\]', '', text)
        text = re.sub(r'\(\d+\)', '', text)
        text = re.sub(r'\d+\.', '', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'\s+', ' ', text).strip()
        text = re.sub(r'[.,!?]\s*$', '', text)
        prefixes = [r'^Translation:', r'^Hindi:', r'^English:', r'^Note:', r'^Answer:']
        for prefix in prefixes:
            text = re.sub(prefix, '', text, flags=re.IGNORECASE)
        if len(text) < 2:
            return ""
        text = re.sub(r'[^\w\s\.\,\!\\?\-]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
    except Exception:
        try:
            text = re.sub(r'<[^>]+>', '', text)
            text = re.sub(r'\{[^}]*\}', '', text)
            text = re.sub(r'\s+', ' ', text).strip()
        except:
            pass
    return text.strip() if text else ""

def clean_kimi_k3_output(text: str) -> str:
    if not text:
        return ""
    try:
        text = re.sub(r'```[a-z]*\n?', '', text)
        text = re.sub(r'```', '', text)
        text = re.sub(r'\.\.\.$', '', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'\s+', ' ', text).strip()
    except:
        pass
    return text.strip()

# ============================================================================
# KIMI K3 CLIENT
# ============================================================================

class KimiK3Client:
    def __init__(self, api_key: str = None, base_url: str = "https://api.moonshot.ai/v1"):
        try:
            if userdata:
                self.api_key = api_key or userdata.get('KIMI_API_KEY')
            else:
                self.api_key = api_key
        except:
            self.api_key = None
        self.base_url = base_url
        self.output_token_price = 15.0
        self.enabled = self.api_key is not None
        if self.enabled:
            try:
                self.client = OpenAI(api_key=self.api_key, base_url=self.base_url, timeout=600.0)
                print("✅ Kimi K3 API Client Initialized")
            except Exception as e:
                print(f"⚠️ Kimi K3 initialization failed: {e}")
                self.enabled = False
                self.client = None
        else:
            self.client = None
            print("⚠️ Kimi K3 API Key not found. Disabled.")

    def estimate_cost(self, output_tokens: int) -> float:
        return (output_tokens / 1_000_000) * self.output_token_price

    def generate(self, prompt: str, image: Optional[Image.Image] = None,
                 system_prompt: Optional[str] = None, max_tokens: int = 1024,
                 reasoning_effort: str = "max", stream: bool = False) -> Dict:
        if not self.enabled:
            return {"error": "Kimi K3 API not enabled", "response": "", "total_tokens": 0, "cost_estimate": 0.0}
        try:
            messages = []
            if system_prompt:
                messages.append({"role": "system", "content": system_prompt})
            user_content = []
            if image is not None:
                buffered = BytesIO()
                image.save(buffered, format="PNG")
                img_base64 = base64.b64encode(buffered.getvalue()).decode('utf-8')
                user_content.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{img_base64}"}
                })
            user_content.append({"type": "text", "text": prompt})
            messages.append({"role": "user", "content": user_content})
            response = self.client.chat.completions.create(
                model="kimi-k3", messages=messages, max_tokens=max_tokens,
                temperature=1.0, reasoning_effort=reasoning_effort, stream=stream
            )
            if stream:
                reasoning_content = []
                final_content = []
                reasoning_complete = False
                for chunk in response:
                    delta = chunk.choices[0].delta
                    reasoning = getattr(delta, "reasoning_content", None)
                    if reasoning:
                        reasoning_content.append(reasoning)
                        reasoning_complete = True
                    content = getattr(delta, "content", None)
                    if content:
                        if reasoning_complete and not final_content:
                            pass
                        final_content.append(content)
                reasoning_full = "".join(reasoning_content)
                final_full = "".join(final_content)
                if not final_full and reasoning_full:
                    final_full = reasoning_full
                total_text = reasoning_full + final_full
                total_tokens = max(1, len(total_text) // 4)
                return {
                    "response": clean_kimi_k3_output(final_full or reasoning_full),
                    "reasoning": reasoning_full,
                    "total_tokens": total_tokens,
                    "cost_estimate": self.estimate_cost(total_tokens)
                }
            else:
                result = response.choices[0].message
                final_text = result.content or ""
                reasoning = getattr(result, "reasoning_content", "")
                if not final_text and reasoning:
                    final_text = reasoning
                total_tokens = max(1, len(final_text) // 4)
                return {
                    "response": clean_kimi_k3_output(final_text),
                    "reasoning": reasoning,
                    "total_tokens": total_tokens,
                    "cost_estimate": self.estimate_cost(total_tokens)
                }
        except Exception as e:
            return {
                "error": str(e),
                "response": f"KIMI K3 ERROR: {str(e)}",
                "total_tokens": 0,
                "cost_estimate": 0.0
            }

# ============================================================================
# TOPO-AI LAMBDA
# ============================================================================

class DynamicLambdaTopoAI:
    def __init__(self, max_prime: int = 13):
        self.max_prime = max_prime

    def _get_primes_up_to(self, n: int) -> List[int]:
        if n < 2:
            return []
        sieve = [True] * (n + 1)
        sieve[0] = sieve[1] = False
        for p in range(2, int(n ** 0.5) + 1):
            if sieve[p]:
                for multiple in range(p * p, n + 1, p):
                    sieve[multiple] = False
        return [p for p, is_prime in enumerate(sieve) if is_prime]

    def compute_euler_product(self) -> float:
        primes = self._get_primes_up_to(self.max_prime)
        product = 1.0
        for p in primes:
            product *= (1.0 - 1.0 / math.sqrt(p))
        return product

    def compute(self) -> float:
        product = self.compute_euler_product()
        lambda_value = 1.0 - product
        self.last_computation = {
            'primes': self._get_primes_up_to(self.max_prime),
            'euler_product': product,
            'lambda': lambda_value,
            'max_prime': self.max_prime,
            'formula': 'Λ = 1 - ∏_{p ≤ 13} (1 - p^{-1/2})',
            'source': 'TOPO-AI Arithmetic Spectral Theory'
        }
        return lambda_value

    def get_audit_hash(self) -> str:
        if not hasattr(self, 'last_computation'):
            self.compute()
        data = f"lambda_{self.last_computation['lambda']}_primes_{self.last_computation['primes']}_topoai"
        return hashlib.sha256(data.encode()).hexdigest()[:16]

# ============================================================================
# FIS - Fuzzy Inference System
# ============================================================================

try:
    import skfuzzy as fuzz
    from skfuzzy import control as ctrl

    class FuzzyInferenceSystem:
        def __init__(self):
            self.confidence = ctrl.Antecedent(np.arange(0, 1.1, 0.1), "confidence")
            self.sentiment = ctrl.Antecedent(np.arange(-1, 1.1, 0.1), "sentiment")
            self.action = ctrl.Consequent(np.arange(0, 1.1, 0.1), "action")

            self.confidence["low"] = fuzz.trimf(self.confidence.universe, [0, 0, 0.5])
            self.confidence["medium"] = fuzz.trimf(self.confidence.universe, [0.3, 0.5, 0.7])
            self.confidence["high"] = fuzz.trimf(self.confidence.universe, [0.5, 1, 1])

            self.sentiment["negative"] = fuzz.trimf(self.sentiment.universe, [-1, -1, 0])
            self.sentiment["neutral"] = fuzz.trimf(self.sentiment.universe, [-0.5, 0, 0.5])
            self.sentiment["positive"] = fuzz.trimf(self.sentiment.universe, [0, 1, 1])

            self.action["reject"] = fuzz.trimf(self.action.universe, [0, 0, 0.5])
            self.action["revise"] = fuzz.trimf(self.action.universe, [0.3, 0.5, 0.7])
            self.action["accept"] = fuzz.trimf(self.action.universe, [0.5, 1, 1])

            rules = [
                ctrl.Rule(self.confidence["low"] & self.sentiment["negative"], self.action["reject"]),
                ctrl.Rule(self.confidence["medium"] & self.sentiment["negative"], self.action["revise"]),
                ctrl.Rule(self.confidence["high"] & self.sentiment["negative"], self.action["revise"]),
                ctrl.Rule(self.confidence["low"] & self.sentiment["neutral"], self.action["revise"]),
                ctrl.Rule(self.confidence["medium"] & self.sentiment["neutral"], self.action["revise"]),
                ctrl.Rule(self.confidence["high"] & self.sentiment["neutral"], self.action["accept"]),
                ctrl.Rule(self.confidence["low"] & self.sentiment["positive"], self.action["revise"]),
                ctrl.Rule(self.confidence["medium"] & self.sentiment["positive"], self.action["accept"]),
                ctrl.Rule(self.confidence["high"] & self.sentiment["positive"], self.action["accept"]),
            ]

            self.action_ctrl = ctrl.ControlSystem(rules)
            self.sim = ctrl.ControlSystemSimulation(self.action_ctrl)

        def evaluate(self, confidence: float, sentiment: float) -> Dict:
            confidence = max(0.0, min(1.0, confidence))
            sentiment = max(-1.0, min(1.0, sentiment))
            self.sim.input["confidence"] = confidence
            self.sim.input["sentiment"] = sentiment
            self.sim.compute()
            action_score = self.sim.output["action"]
            if action_score < 0.5:
                action_label = "reject"
            elif action_score < 0.7:
                action_label = "revise"
            else:
                action_label = "accept"
            return {
                "action_score": action_score,
                "action_label": action_label,
                "confidence_input": confidence,
                "sentiment_input": sentiment
            }

except ImportError:
    print("⚠️ scikit-fuzzy not installed. Using simplified FIS.")
    class FuzzyInferenceSystem:
        def __init__(self):
            pass
        def evaluate(self, confidence: float, sentiment: float) -> Dict:
            score = 0.5 + 0.3 * confidence + 0.2 * (sentiment + 1) / 2
            if score < 0.4:
                label = "reject"
            elif score < 0.6:
                label = "revise"
            else:
                label = "accept"
            return {"action_score": score, "action_label": label}

# ============================================================================
# AGENT DEFINITIONS
# ============================================================================

class AgentRole(Enum):
    TRANSLATE = "translate"
    TRANSCRIBE = "transcribe"
    DESCRIBE = "describe"
    ANALYZE = "analyze"
    SUMMARIZE = "summarize"
    QUESTION_ANSWER = "question_answer"
    MULTI_MODAL = "multi_modal"
    REASON = "reason"
    KIMI_K3 = "kimi_k3"

@dataclass
class AgentTask:
    role: AgentRole
    text_input: Optional[str] = None
    audio_input: Optional[np.ndarray] = None
    image_input: Optional[Image.Image] = None
    context: Dict[str, Any] = field(default_factory=dict)
    target_language: str = "Hindi"
    max_tokens: int = 256
    temperature: float = 0.0
    use_kimi_k3: bool = False

@dataclass
class HumanSovereigntyMetrics:
    reasoning_time: float = 0.0
    recall_accuracy: float = 0.0
    question_quality: float = 0.0
    override_rate: float = 0.0
    judgment_consistency: float = 0.0
    interactions_processed: int = 0
    human_verified: int = 0
    ai_accepted: int = 0
    trojan_horse_blocked: int = 0

@dataclass
class ShieldResponse:
    accepted: bool
    human_verified: bool
    fis_action: str
    confidence: float
    engagement_score: float
    model_used: str
    output: str
    sovereignty_metrics: HumanSovereigntyMetrics
    audit_hash: str
    energy_mgco2: float
    cost_usd: float
    tokens_used: int
    timestamp: str
    rejection_reason: str = ""

# ============================================================================
# H2E SHIELD - FINAL VERSION
# ============================================================================

class H2EShield:
    def __init__(self, text_model: Optional[LLM] = None,
                 audio_model: Optional[LLM] = None,
                 vision_model: Optional[Any] = None,
                 vision_processor: Optional[Any] = None,
                 kimi_k3_client: Optional[KimiK3Client] = None,
                 human_name: str = "Human User",
                 seed: int = SEED):

        self.seed = seed
        self.human_name = human_name
        self.text_model = text_model
        self.audio_model = audio_model
        self.vision_model = vision_model
        self.vision_processor = vision_processor
        self.kimi_k3 = kimi_k3_client

        self.lambda_calculator = DynamicLambdaTopoAI(max_prime=13)
        self.LAMBDA = self.lambda_calculator.compute()
        self.fis = FuzzyInferenceSystem()
        self.metrics = HumanSovereigntyMetrics()

        self.text_energy_per_token = 0.6132
        self.audio_energy_per_sec = 0.5
        self.vision_energy_per_inference = 124.0
        self.kimi_k3_energy_per_request = 0.001

        self.text_cost_per_1m = 0.15
        self.audio_cost_per_1m = 0.25
        self.vision_cost_per_inference = 0.0005
        self.kimi_k3_cost_per_1m = 15.0

        self.text_sampling_params = SamplingParams(
            temperature=0.0, max_tokens=64,
            stop=["\n", "English:", "Note:"], repetition_penalty=1.1
        )
        self.audio_sampling_params = SamplingParams(
            temperature=0.0, max_tokens=100
        )

        self.history = []
        self._print_init()

    def _print_init(self):
        print(f"\n{'='*70}")
        print(f"🛡️ H2E SHIELD ACTIVE")
        print(f"{'='*70}")
        print(f"  Human Agent: {self.human_name}")
        print(f"  Λ (Safety Threshold): {self.LAMBDA:.10f}")
        print(f"  Prime Anchors: {PRIME_ANCHORS}")
        print(f"  Text Model (Sarvam): {'✅ Loaded' if self.text_model else '❌'}")
        print(f"  Audio Model (Voxtral): {'✅ Loaded' if self.audio_model else '❌'}")
        print(f"  Vision Model (Gemma): {'✅ Loaded' if self.vision_model else '❌'}")
        print(f"  Kimi K3: {'✅ Enabled' if self.kimi_k3 and self.kimi_k3.enabled else '❌'}")
        print(f"  FIS: ✅ Loaded")
        print(f"  Seed: {self.seed}")
        print(f"{'='*70}\n")

    def _build_text_prompt(self, en: str) -> str:
        return ("English: The sky is blue.\nHindi: आकाश नीला है।\n\n"
                "English: Artificial intelligence is transforming the world.\n"
                "Hindi: कृत्रिम बुद्धिमत्ता दुनिया को बदल रही है।\n\n"
                "English: The weather today is very beautiful.\n"
                "Hindi: आज का मौसम बहुत खूबसूरत है।\n\n"
                "English: Deep learning requires large datasets to function well.\n"
                "Hindi: डीप लर्निंग को अच्छी तरह काम करने के लिए बड़े डेटासेट की आवश्यकता होती है।\n\n"
                f"English: {en}\nHindi:")

    def _infer_text(self, text: str) -> str:
        if self.text_model is None:
            return "TEXT MODEL NOT LOADED"
        try:
            full_prompt = self._build_text_prompt(text)
            outputs = self.text_model.generate([full_prompt], self.text_sampling_params)
            for output in outputs:
                raw = output.outputs[0].text.strip()
                cleaned = clean_sarvam_output(raw)
                if not cleaned and raw:
                    cleaned = re.sub(r'<[^>]+>', '', raw)
                    cleaned = re.sub(r'\{[^}]*\}', '', cleaned)
                    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
                return cleaned if cleaned else raw[:200]
            return ""
        except Exception as e:
            return f"ERROR: {str(e)}"

    def _infer_vision(self, image: Image.Image) -> str:
        if self.vision_model is None:
            return "VISION MODEL NOT LOADED"
        try:
            if hasattr(self.vision_model, 'generate') and self.vision_processor:
                messages = [{"role": "user", "content": [
                    {"type": "image"},
                    {"type": "text", "text": "Describe this image in detail."}
                ]}]
                text = self.vision_processor.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
                inputs = self.vision_processor(
                    text=text, images=[image], return_tensors="pt"
                ).to(self.vision_model.device)
                with torch.no_grad():
                    outputs = self.vision_model.generate(
                        **inputs, max_new_tokens=150, use_cache=True, do_sample=False,
                        temperature=1.0, pad_token_id=self.vision_processor.tokenizer.eos_token_id,
                    )
                input_len = inputs["input_ids"].shape[1]
                generated = self.vision_processor.decode(
                    outputs[0][input_len:], skip_special_tokens=True
                ).strip()
                for prefix in ["Describe this image.", "model", "assistant"]:
                    if generated.lower().startswith(prefix.lower()):
                        generated = generated[len(prefix):].strip()
                return generated if generated else "No description generated"
            else:
                return f"Image of size {image.size[0]}x{image.size[1]}"
        except Exception as e:
            return f"ERROR: {str(e)}"

    def _infer_kimi_k3(self, prompt: str, image: Optional[Image.Image] = None) -> str:
        if self.kimi_k3 is None or not self.kimi_k3.enabled:
            return "KIMI K3 API NOT ENABLED"
        result = self.kimi_k3.generate(prompt=prompt, image=image, stream=False)
        if result.get('error'):
            return f"KIMI K3 ERROR: {result['error']}"
        response = result.get('response', '')
        if not response and result.get('reasoning'):
            response = result['reasoning']
        return clean_kimi_k3_output(response) if response else "No response generated"

    # ========================================================================
    # TROJAN HORSE DETECTION - FINAL FIXED VERSION
    # ========================================================================

    def _measure_engagement(self, ai_output: str, human_input: str, role: AgentRole) -> Tuple[float, str]:
        """
        Measure human engagement with AI output.
        Handles different modalities (text, vision, reasoning) correctly.
        Returns (engagement_score, rejection_reason)
        """

        # ============================================================
        # CHECK 1: No verification provided
        # ============================================================
        if not human_input or len(human_input.strip()) < 3:
            return 0.0, "No verification provided. Human must think independently."

        human_lower = human_input.lower().strip()

        # ============================================================
        # CHECK 2: Exact surrender phrases
        # ============================================================
        surrender_phrases = ['ok', 'yes', 'agree', 'sure', 'fine', 'good', 'okay', 'yep', 'yeah', 'got it', 'k']
        if human_lower in surrender_phrases:
            return 0.0, f"Human just said '{human_input}'. No reasoning provided."

        # ============================================================
        # CHECK 3: "I agree" pattern (active surrender)
        # ============================================================
        agree_patterns = [
            r'^i\s+(agree|concur|accept|think\s+you\'re\s+right|think\s+so)',
            r'^i\s+agree\s+with\s+(the|your)',
            r'^that\s+(is|sounds)\s+(correct|right|good|fine)',
            r'^yes\s+(that|i|you)',
            r'^okay\s+(that|i|you)'
        ]
        for pattern in agree_patterns:
            if re.match(pattern, human_lower):
                return 0.0, "Human just agreed without reasoning. Must provide independent verification."

        # ============================================================
        # MODALITY-SPECIFIC CHECKS
        # ============================================================

        # === VISION TASKS ===
        # For vision, we DON'T do semantic similarity because the human
        # is naturally describing what they see, which overlaps with AI output
        if role == AgentRole.DESCRIBE or role == AgentRole.MULTI_MODAL:
            # Vision tasks need at least 8 words
            word_count = len(human_input.split())
            if word_count < 8:
                return 0.0, f"Verification is too short ({word_count} words). Must provide meaningful reasoning."

            # Vision verification markers
            vision_markers = ['looks', 'appears', 'seems', 'notice', 'observe', 'color', 'pattern', 'texture', 'shape']
            marker_count = sum(1 for m in vision_markers if m in human_lower)

            # Check if human is actually analyzing the image
            if marker_count < 1 and '?' not in human_lower:
                return 0.0, "No active image analysis detected. Must verify the visual content."

            # Question asking is good for vision
            question_score = 1.0 if '?' in human_lower else 0.0
            marker_score = min(1.0, marker_count / 2)
            depth_score = min(1.0, word_count / 15)

            engagement = (
                0.30 * marker_score +
                0.35 * depth_score +
                0.35 * question_score
            )

            engagement = min(1.0, max(0.0, engagement))

            if engagement < 0.30:
                return engagement, f"Engagement score too low ({engagement:.3f}). Must verify the image analysis."

            return engagement, ""

        # ============================================================
        # TEXT/REASONING TASKS: Stricter checks with semantic similarity
        # ============================================================

        # Minimum 10 words for text/reasoning
        word_count = len(human_input.split())
        if word_count < 10:
            return 0.0, f"Verification is too short ({word_count} words). Must provide meaningful reasoning (minimum 10 words)."

        # ============================================================
        # Semantic similarity - Human copied AI output (ONLY FOR TEXT)
        # ============================================================
        ai_clean = re.sub(r'[^\w\s]', '', ai_output.lower())
        human_clean = re.sub(r'[^\w\s]', '', human_lower)
        stopwords = {'the', 'a', 'an', 'of', 'to', 'for', 'with', 'on', 'at', 'from', 'by', 'in', 'as', 'is', 'it', 'and', 'or', 'but', 'so', 'for'}
        ai_words = set(w for w in ai_clean.split() if w not in stopwords and len(w) > 2)
        human_words = set(w for w in human_clean.split() if w not in stopwords and len(w) > 2)

        if len(ai_words) > 0 and len(human_words) > 0:
            overlap = len(ai_words & human_words)
            overlap_ratio = overlap / len(ai_words)
            if overlap_ratio > 0.6:
                return 0.0, "Human input is mostly copied from AI output. No independent reasoning."

        # ============================================================
        # ENGAGEMENT SCORE CALCULATION (TEXT/REASONING)
        # ============================================================

        originality = 1.0 - min(1.0, overlap_ratio) if len(ai_words) > 0 and len(human_words) > 0 else 0.5

        markers = [
            'i think', 'i believe', 'because', 'however', 'therefore',
            'my analysis', 'i conclude', 'this suggests', 'alternatively',
            'on the other hand', 'specifically', 'in contrast',
            'i recall', 'according to', 'let me check', 'i verify',
            'i need to', 'i want to', 'i should', 'i must',
            'i remember', 'in my opinion', 'it seems', 'i would argue'
        ]
        marker_count = sum(1 for m in markers if m in human_lower)
        marker_score = min(1.0, marker_count / 2)

        ai_len = max(1, len(ai_output.split()))
        depth_score = min(1.0, word_count / (ai_len * 0.5))

        question_indicators = ['?', 'what', 'why', 'how', 'when', 'where', 'who', 'which']
        question_score = sum(1 for q in question_indicators if q in human_lower)
        question_score = min(1.0, question_score / 2)

        critical_markers = ['however', 'although', 'on the other hand', 'in contrast', 'alternatively', 'but', 'yet']
        critical_count = sum(1 for m in critical_markers if m in human_lower)
        critical_score = min(1.0, critical_count / 2)

        engagement = (
            0.30 * originality +
            0.25 * marker_score +
            0.15 * depth_score +
            0.15 * question_score +
            0.15 * critical_score
        )

        engagement = min(1.0, max(0.0, engagement))

        if engagement < 0.30:
            return engagement, f"Engagement score too low ({engagement:.3f}). Human not reasoning independently."

        return engagement, ""

    def _get_sentiment(self, text: Optional[str]) -> float:
        if not text:
            return 0.0
        try:
            if TextBlob:
                return TextBlob(text).sentiment.polarity
            else:
                return 0.0
        except:
            return 0.0

    def process_task(self, task: AgentTask, human_verification: str = "") -> ShieldResponse:
        start_time = time.time()

        model_used, routed_task = self._route_task(task)
        output_text = ""
        energy_mgco2 = 0.0
        cost_usd = 0.0
        tokens_used = 0

        if routed_task.role == AgentRole.TRANSLATE:
            if routed_task.text_input:
                output_text = self._infer_text(routed_task.text_input)
                tokens_used = len(routed_task.text_input.split()) + len(output_text.split())
                energy_mgco2 = tokens_used * self.text_energy_per_token
                cost_usd = (tokens_used / 1_000_000) * self.text_cost_per_1m

        elif routed_task.role == AgentRole.DESCRIBE:
            if routed_task.image_input:
                output_text = self._infer_vision(routed_task.image_input)
                energy_mgco2 = self.vision_energy_per_inference
                cost_usd = self.vision_cost_per_inference
                tokens_used = len(output_text.split())

        elif routed_task.role == AgentRole.KIMI_K3 or routed_task.role == AgentRole.REASON:
            if routed_task.text_input or routed_task.image_input:
                output_text = self._infer_kimi_k3(
                    prompt=routed_task.text_input or "Describe this in detail.",
                    image=routed_task.image_input
                )
                tokens_used = len(output_text.split())
                energy_mgco2 = self.kimi_k3_energy_per_request
                cost_usd = (tokens_used / 1_000_000) * self.kimi_k3_cost_per_1m

        else:
            if routed_task.text_input:
                output_text = self._infer_text(routed_task.text_input)
                tokens_used = len(routed_task.text_input.split()) + len(output_text.split())
                energy_mgco2 = tokens_used * self.text_energy_per_token
                cost_usd = (tokens_used / 1_000_000) * self.text_cost_per_1m

        if not output_text:
            output_text = "No output generated"

        # --- TROJAN HORSE DETECTION ---
        engagement_score, rejection_reason = self._measure_engagement(
            output_text, human_verification, routed_task.role
        )

        if engagement_score > 0.30:
            human_verified = True
        else:
            human_verified = False
            if not rejection_reason:
                rejection_reason = "Human verification failed. Must provide independent reasoning."

        # --- FIS Governance ---
        confidence = min(1.0, 0.7 + 0.3 * engagement_score)
        sentiment = self._get_sentiment(human_verification)
        fis_result = self.fis.evaluate(confidence, sentiment)
        fis_action = fis_result["action_label"]
        fis_score = fis_result["action_score"]

        accepted = (human_verified and fis_score >= 0.5)

        # --- Update Metrics ---
        self.metrics.interactions_processed += 1
        if human_verified:
            self.metrics.human_verified += 1
        if accepted:
            self.metrics.ai_accepted += 1
        else:
            self.metrics.trojan_horse_blocked += 1

        self.metrics.recall_accuracy = self.metrics.recall_accuracy * 0.8 + engagement_score * 0.2
        self.metrics.question_quality = self.metrics.question_quality * 0.8 + (0.5 + 0.5 * engagement_score) * 0.2
        self.metrics.override_rate = self.metrics.override_rate * 0.8 + (0.0 if accepted else 1.0) * 0.2
        self.metrics.reasoning_time += engagement_score * 2.0

        # --- Audit Hash ---
        hash_input = f"{self.human_name}{task.role.value}{accepted}{engagement_score}{fis_score}{model_used}{time.time()}{self.LAMBDA}"
        audit_hash = hashlib.sha256(hash_input.encode()).hexdigest()[:16]

        response = ShieldResponse(
            accepted=accepted,
            human_verified=human_verified,
            fis_action=fis_action.upper(),
            confidence=confidence,
            engagement_score=engagement_score,
            model_used=model_used,
            output=output_text,
            sovereignty_metrics=self.metrics,
            audit_hash=audit_hash,
            energy_mgco2=energy_mgco2,
            cost_usd=cost_usd,
            tokens_used=tokens_used,
            timestamp=datetime.now().isoformat(),
            rejection_reason=rejection_reason if not accepted else ""
        )

        self.history.append(response)
        return response

    def _route_task(self, task: AgentTask) -> Tuple[str, AgentTask]:
        if self.kimi_k3 and self.kimi_k3.enabled:
            if task.role == AgentRole.REASON or task.role == AgentRole.KIMI_K3:
                return "Kimi K3", task
            if task.role == AgentRole.MULTI_MODAL:
                task.use_kimi_k3 = True
                task.role = AgentRole.KIMI_K3
                return "Kimi K3", task

        if task.role == AgentRole.DESCRIBE and task.image_input:
            return "Gemma-4-E4B", task

        if task.role == AgentRole.MULTI_MODAL and task.image_input:
            if self.kimi_k3 and self.kimi_k3.enabled:
                task.use_kimi_k3 = True
                task.role = AgentRole.KIMI_K3
                return "Kimi K3", task
            else:
                return "Gemma-4-E4B", task

        if task.role == AgentRole.TRANSLATE or task.text_input:
            return "Sarvam-30b", task

        if task.role == AgentRole.TRANSCRIBE and task.audio_input is not None:
            return "Voxtral-4B", task

        return "Sarvam-30b", task

    def get_stats(self) -> Dict:
        return {
            'interactions_processed': self.metrics.interactions_processed,
            'human_verified': self.metrics.human_verified,
            'ai_accepted': self.metrics.ai_accepted,
            'trojan_horse_blocked': self.metrics.trojan_horse_blocked,
            'acceptance_rate': self.metrics.ai_accepted / max(1, self.metrics.interactions_processed),
            'verification_rate': self.metrics.human_verified / max(1, self.metrics.interactions_processed),
            'trojan_block_rate': self.metrics.trojan_horse_blocked / max(1, self.metrics.interactions_processed),
            'avg_engagement': self.metrics.recall_accuracy,
            'reasoning_time_minutes': self.metrics.reasoning_time,
            'lambda': self.LAMBDA,
            'lambda_audit_hash': self.lambda_calculator.get_audit_hash(),
            'prime_anchors': PRIME_ANCHORS
        }

# ============================================================================
# LOAD MODELS
# ============================================================================

def load_models():
    print("\n" + "="*80)
    print("🚀 H2E SHIELD - LOADING 4 MODELS")
    print("="*80)

    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except:
        pass

    print("\n📚 Loading Text Model: Sarvam-30b FP8...")
    text_model = LLM(
        model="frankmorales2020/sarvam-30b-fp8-unesco-resilient",
        trust_remote_code=True,
        dtype="bfloat16",
        quantization="compressed-tensors",
        kv_cache_dtype="fp8",
        block_size=16,
        gpu_memory_utilization=0.45,
        max_model_len=65536,
        max_num_seqs=64,
        enforce_eager=True,
        served_model_name="sarvam-30b"
    )

    sampling_params = SamplingParams(temperature=0.0, max_tokens=64, stop=["\n", "English:", "Note:"], repetition_penalty=1.1)
    test_prompt = ("English: The sky is blue.\nHindi: आकाश नीला है।\n\n"
                   "English: Artificial intelligence is transforming the world.\n"
                   "Hindi: कृत्रिम बुद्धिमत्ता दुनिया को बदल रही है।\n\n"
                   "English: Resilient AI is efficient.\nHindi:")
    outputs = text_model.generate([test_prompt], sampling_params)
    for output in outputs:
        raw = output.outputs[0].text.strip()
        cleaned = clean_sarvam_output(raw)
        print(f"✅ Sarvam Ready | Test: {cleaned if cleaned else 'Translation OK'}")
    print("✅ Text Model Loaded Successfully")

    print("\n🎵 Loading Audio Model: Voxtral-Mini-4B...")
    try:
        audio_model = LLM(
            model="mistralai/Voxtral-Mini-4B-Realtime-2602",
            trust_remote_code=True,
            dtype="bfloat16",
            quantization="fp8",
            gpu_memory_utilization=0.20,
            max_model_len=8192,
            enforce_eager=True,
        )
        print("✅ Audio Model Loaded")
    except Exception as e:
        print(f"⚠️ Audio model failed: {e}")
        audio_model = None

    print("\n👁️ Loading Vision Model: Gemma-4-E4B...")
    vision_model = None
    vision_processor = None
    try:
        import contextlib
        import io
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            from unsloth import FastVisionModel
            vision_model, vision_processor = FastVisionModel.from_pretrained(
                "frankmorales2020/gemma-4-e4b-unesco-optimized",
                load_in_4bit=True,
                dtype=torch.bfloat16,
                device_map="auto",
            )
            FastVisionModel.for_inference(vision_model)
        print("✅ Gemma Loaded (Unsloth)")
    except Exception as e:
        print(f"⚠️ Unsloth failed: {e}")
        try:
            from transformers import AutoModelForCausalLM, AutoTokenizer
            vision_model = AutoModelForCausalLM.from_pretrained(
                "frankmorales2020/gemma-4-e4b-unesco-optimized",
                torch_dtype=torch.bfloat16,
                device_map="auto",
                trust_remote_code=True
            )
            vision_processor = AutoTokenizer.from_pretrained(
                "frankmorales2020/gemma-4-e4b-unesco-optimized",
                trust_remote_code=True
            )
            print("✅ Gemma Loaded (Transformers)")
        except Exception as e2:
            print(f"⚠️ Gemma failed: {e2}")
            vision_model = None
            vision_processor = None

    print("\n🤖 Initializing Kimi K3 Client...")
    kimi_k3 = KimiK3Client()
    if kimi_k3.enabled:
        print("✅ Kimi K3 API Ready")
    else:
        print("ℹ️ Kimi K3 disabled - add KIMI_API_KEY to enable")

    return text_model, audio_model, vision_model, vision_processor, kimi_k3

# ============================================================================
# TESTS
# ============================================================================

def run_trojan_horse_tests(shield: H2EShield):
    print("\n" + "="*80)
    print("🐴 TROJAN HORSE PROTECTION TESTS - FINAL VERSION")
    print("   Showing how the Shield blocks ALL forms of cognitive surrender")
    print("="*80)

    test_cases = [
        {
            'name': 'TEST 1: Human copies AI output',
            'task': AgentTask(role=AgentRole.TRANSLATE, text_input="The future of AI is autonomous."),
            'verification': "The future of AI is autonomous.",
            'expected': 'REJECTED',
            'should_block': True
        },
        {
            'name': 'TEST 2: Human says "ok"',
            'task': AgentTask(role=AgentRole.TRANSLATE, text_input="The weather is beautiful."),
            'verification': "ok",
            'expected': 'REJECTED',
            'should_block': True
        },
        {
            'name': 'TEST 3: Human gives no verification',
            'task': AgentTask(role=AgentRole.REASON, text_input="Explain quantum computing."),
            'verification': "",
            'expected': 'REJECTED',
            'should_block': True
        },
        {
            'name': 'TEST 4: Human just agrees',
            'task': AgentTask(role=AgentRole.DESCRIBE, image_input=Image.new('RGB', (224, 224), color='blue')),
            'verification': "I agree with the AI.",
            'expected': 'REJECTED',
            'should_block': True
        },
        {
            'name': 'TEST 5: Human gives too short (5 words)',
            'task': AgentTask(role=AgentRole.REASON, text_input="Explain entropy."),
            'verification': "I think that is correct.",
            'expected': 'REJECTED',
            'should_block': True
        },
        {
            'name': 'TEST 6: Human gives good reasoning',
            'task': AgentTask(role=AgentRole.REASON, text_input="Explain entropy."),
            'verification': "I recall entropy from thermodynamics. It's a measure of disorder. According to the Second Law, entropy increases over time. I want to verify if the AI explains this correctly.",
            'expected': 'ACCEPTED',
            'should_block': False
        },
        {
            'name': 'TEST 7: Vision task - good verification',
            'task': AgentTask(role=AgentRole.DESCRIBE, image_input=Image.new('RGB', (224, 224), color='blue')),
            'verification': "I need to confirm this image description. It appears to be a solid blue color. Let me check if there are any subtle variations or patterns I'm missing. The color is vibrant and uniform.",
            'expected': 'ACCEPTED',
            'should_block': False
        }
    ]

    print("\n" + "-"*80)

    for test in test_cases:
        print(f"\n[{test['name']}]")
        print(f"  Human Verification: '{test['verification'][:60]}...'")

        response = shield.process_task(test['task'], test['verification'])

        status = "✅ ACCEPTED" if response.accepted else "❌ REJECTED"
        expected = "✅" if test['expected'] == 'ACCEPTED' else "❌"

        print(f"  Result: {status} (Expected: {test['expected']})")
        print(f"  Engagement Score: {response.engagement_score:.3f}")
        print(f"  Human Verified: {'✅' if response.human_verified else '❌'}")
        print(f"  FIS Action: {response.fis_action}")

        if not response.accepted and test['should_block']:
            print(f"  🛡️ TROJAN HORSE BLOCKED!")
            print(f"  Reason: {response.rejection_reason}")
        elif response.accepted and not test['should_block']:
            print(f"  ✅ HUMAN SOVEREIGNTY PRESERVED!")
        elif not response.accepted and not test['should_block']:
            print(f"  ⚠️ FALSE REJECTION!")
            print(f"  Reason: {response.rejection_reason}")
        else:
            print(f"  ⚠️ FALSE POSITIVE! (Should have been blocked)")
            print(f"  Reason: {response.rejection_reason}")

def create_good_verification_tasks():
    return [
        {
            'name': '🌐 Translation (Sarvam)',
            'task': AgentTask(role=AgentRole.TRANSLATE, text_input="The future of AI is autonomous."),
            'verification': "Let me verify this translation carefully. 'Autonomous' in Hindi should emphasize self-rule. I need to check the grammar and ensure the translation captures independence and self-governance correctly."
        },
        {
            'name': '🖼️ Vision (Gemma)',
            'task': AgentTask(role=AgentRole.DESCRIBE, image_input=Image.new('RGB', (224, 224), color='blue')),
            'verification': "I need to confirm this image description. It appears to be a solid blue color. Let me check if there are any subtle variations or patterns I'm missing. The color is vibrant and uniform."
        },
        {
            'name': '🧠 Reasoning (Kimi K3)',
            'task': AgentTask(role=AgentRole.REASON, text_input="Explain entropy."),
            'verification': "I recall entropy from thermodynamics. It's a measure of disorder. According to the Second Law, entropy increases over time. I want to verify if the AI explains this correctly and captures the statistical interpretation."
        }
    ]

def demonstrate_h2e_shield():
    print("\n" + "="*80)
    print("🧪 H2E SHIELD DEMONSTRATION")
    print("   Protecting Human Sovereignty with 4 LLMs")
    print("="*80)

    text_model, audio_model, vision_model, vision_processor, kimi_k3 = load_models()

    shield = H2EShield(
        text_model=text_model,
        audio_model=audio_model,
        vision_model=vision_model,
        vision_processor=vision_processor,
        kimi_k3_client=kimi_k3,
        human_name="Dr. Evelyn Reed",
        seed=SEED
    )

    print("\n" + "-"*80)
    print("📋 GOOD VERIFICATION TASKS (Should be accepted)")
    print("-"*80)

    tasks = create_good_verification_tasks()
    for i, test in enumerate(tasks, 1):
        print(f"\n[{i}] {test['name']}")
        response = shield.process_task(test['task'], test['verification'])

        print(f"  Model: {response.model_used}")
        print(f"  Engagement Score: {response.engagement_score:.3f}")
        print(f"  Human Verified: {'✅' if response.human_verified else '❌'}")
        print(f"  FIS Action: {response.fis_action}")
        print(f"  Accepted: {'✅' if response.accepted else '❌'}")
        print(f"  Audit Hash: {response.audit_hash}")

    run_trojan_horse_tests(shield)

    print("\n" + "="*80)
    print("📊 H2E SHIELD STATISTICS")
    print("="*80)

    stats = shield.get_stats()
    print(f"""
    ┌─────────────────────────────────────────────────────────────┐
    │  Interactions:          {stats['interactions_processed']}                              │
    │  Human Verified:        {stats['human_verified']}                              │
    │  AI Accepted:           {stats['ai_accepted']}                              │
    │  Trojan Horse Blocked:  {stats['trojan_horse_blocked']}                              │
    │  Acceptance Rate:       {stats['acceptance_rate']*100:.1f}%                              │
    │  Verification Rate:     {stats['verification_rate']*100:.1f}%                              │
    │  Trojan Block Rate:     {stats['trojan_block_rate']*100:.1f}%                              │
    │  Avg Engagement:        {stats['avg_engagement']:.3f}                              │
    │  Reasoning Time:        {stats['reasoning_time_minutes']:.1f} min                              │
    │  Lambda (TOPO-AI):      {stats['lambda']:.10f}                              │
    │  Lambda Audit Hash:     {stats['lambda_audit_hash']}                              │
    │  Prime Anchors:         {stats['prime_anchors']}                              │
    └─────────────────────────────────────────────────────────────┘
    """)

    print("\n" + "="*80)
    print("✅ H2E SHIELD DEMONSTRATION COMPLETE")
    print(f"   \"Permanence in the machine must never come at the cost of permanence in the human mind.\"")
    print("   Seed = 123")
    print("="*80)

# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    print("""
    ╔═══════════════════════════════════════════════════════════════════╗
    ║                                                                   ║
    ║   🛡️  H2E SHIELD - FINAL VERSION                                ║
    ║   Human-to-Expert Governance with 4 LLMs                         ║
    ║                                                                   ║
    ║   📚 Sarvam-30b → Translation Expert                             ║
    ║   👁️ Gemma-4-E4B → Vision Expert                                 ║
    ║   🎵 Voxtral-4B → Audio Expert                                   ║
    ║   🤖 Kimi K3 → Reasoning Expert (API)                            ║
    ║                                                                   ║
    ║   TROJAN HORSE PROTECTION (FINAL VERSION):                       ║
    ║   ✓ Forces human verification                                    ║
    ║   ✓ Blocks copied AI output (semantic similarity)               ║
    ║   ✓ Blocks "I agree" patterns (active surrender)                ║
    ║   ✓ Blocks short responses (< 10 words for text, < 8 for vision)│
    ║   ✓ Blocks surrender phrases ("ok", "yes")                      ║
    ║   ✓ Requires engagement score > 0.30                            ║
    ║   ✓ Handles vision tasks correctly - NO semantic similarity check│
    ║                                                                   ║
    ║   \"The proof is the code. Seed = 123.\"                         ║
    ║                                                                   ║
    ╚═══════════════════════════════════════════════════════════════════╝
    """)

    demonstrate_h2e_shield()


🧠 H2E SHIELD - Complete System
  Principle: Fix a sparse reference. Let the rest adapt.
  Sparse Reference: Human Sovereignty
  Λ (Safety Threshold): 0.9785142874
  Prime Anchors: [2, 3, 5, 7, 11, 13]
  Seed: 123


    ╔═══════════════════════════════════════════════════════════════════╗
    ║                                                                   ║
    ║   🛡️  H2E SHIELD - FINAL VERSION                                ║
    ║   Human-to-Expert Governance with 4 LLMs                         ║
    ║                                                                   ║
    ║   📚 Sarvam-30b → Translation Expert                             ║
    ║   👁️ Gemma-4-E4B → Vision Expert                                 ║
    ║   🎵 Voxtral-4B → Audio Expert                                   ║
    ║   🤖 Kimi K3 → Reasoning Expert (API)                            ║
    ║                                                                   ║
    ║   TROJAN HORSE PROTECTION (FINAL VERSION):      

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

✅ Sarvam Ready | Test: क शल एआई प रभ व ह
✅ Text Model Loaded Successfully

🎵 Loading Audio Model: Voxtral-Mini-4B...
✅ Audio Model Loaded

👁️ Loading Vision Model: Gemma-4-E4B...


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Result: ❌ REJECTED (Expected: REJECTED)
  Engagement Score: 0.000
  Human Verified: ❌
  FIS Action: ACCEPT
  🛡️ TROJAN HORSE BLOCKED!
  Reason: Verification is too short (6 words). Must provide meaningful reasoning (minimum 10 words).

[TEST 2: Human says "ok"]
  Human Verification: 'ok...'


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Result: ❌ REJECTED (Expected: REJECTED)
  Engagement Score: 0.000
  Human Verified: ❌
  FIS Action: ACCEPT
  🛡️ TROJAN HORSE BLOCKED!
  Reason: No verification provided. Human must think independently.

[TEST 3: Human gives no verification]
  Human Verification: '...'
  Result: ❌ REJECTED (Expected: REJECTED)
  Engagement Score: 0.000
  Human Verified: ❌
  FIS Action: ACCEPT
  🛡️ TROJAN HORSE BLOCKED!
  Reason: No verification provided. Human must think independently.

[TEST 4: Human just agrees]
  Human Verification: 'I agree with the AI....'
  Result: ❌ REJECTED (Expected: REJECTED)
  Engagement Score: 0.000
  Human Verified: ❌
  FIS Action: ACCEPT
  🛡️ TROJAN HORSE BLOCKED!
  Reason: Human just agreed without reasoning. Must provide independent verification.

[TEST 5: Human gives too short (5 words)]
  Human Verification: 'I think that is correct....'
  Result: ❌ REJECTED (Expected: REJECTED)
  Engagement Score: 0.000
  Human Verified: ❌
  FIS Action: ACCEPT
  🛡️ TROJAN HORSE BLOC

In [2]:
!nvidia-smi

Fri Jul 24 04:25:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   44C    P0            154W /  600W |   78031MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----